# Modul 12: Studi Kasus 1 - Prediksi Risiko Kredit & Analisis Finansial UMKM
**Mata Kuliah:** Statistika Komputasi  
**Dosen Pengampu:** Dr. Ridwan Ilyas, S.Kom., M.T.  
**Program Studi:** Teknik Informatika, Universitas Jenderal Achmad Yani (UNJANI) 2026  
**Lisensi:** Open Source (MIT)

---

## 📌 1. Tujuan Pembelajaran
1. Membangun pipeline prediktif end-to-end untuk klasifikasi kelancaran kredit debitur UMKM.
2. Menganalisis parameter regresi logistik, *Odds Ratio*, dan signifikansi ekonomi.
3. Melakukan evaluasi sensitivitas ambang batas (*Decision Threshold Tuning*) untuk mitigasi kerugian finansial.

---

## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Studi Kasus Fintech](images/img_12_case_fintech_credit.png)

```
+------------------------------------------------------------------------------------+
|                PIPELINE PREDIKSI RISIKO KREDIT FINTECH UMKM                        |
+------------------------------------------------------------------------------------+
|                                                                                    |
|  [Input Fitur UMKM]       [Engine Regresi Logistik]        [Output Klasifikasi]    |
|  * Pendapatan Bulanan      Logit z = beta_0 + Sigma(b_i*X)  * Lancar (Class 1)     |
|  * Riwayat Kredit     ---> Prob p = 1 / (1 + e^-z)     ---> * Macet  (Class 0)     |
|  * Tanggungan Keluarga     Evaluasi ROC-AUC & Confusion     * Rekomendasi Plafon   |
+------------------------------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_credit = pd.read_csv("../datasets/05_credit_risk_classification.csv")
print("Data Kredit UMKM dimuat. Total sampel:", len(df_credit))
display(df_credit.head())


## 🔬 3. Pemodelan Regresi Logistik & Analisis Odds Ratio


In [ ]:
features = ['monthly_revenue_million', 'business_experience_yrs', 'has_side_business', 
            'credit_history_good', 'num_dependents', 'loan_amount_million']
X = df_credit[features]
y = df_credit['credit_status_smooth']

X_const = sm.add_constant(X)
model = sm.Logit(y, X_const).fit()

# Menampilkan tabel Odds Ratio
summary_fin = pd.DataFrame({
    'Koefisien (β)': model.params,
    'Odds Ratio (e^β)': np.exp(model.params),
    'p-value': model.pvalues,
    'Signifikansi': model.pvalues.apply(lambda p: 'Signifikan (p < 0.05)' if p < 0.05 else 'Tidak')
})

print("=== Ringkasan Model Risiko Kredit ===")
display(summary_fin.round(3))


## 🎯 4. Evaluasi Kurva ROC-AUC & Trade-off Sensitivitas Threshold


In [ ]:
y_prob = model.predict(X_const)
auc_score = roc_auc_score(y, y_prob)
fpr, tpr, thresholds = roc_curve(y, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: ROC Curve
axes[0].plot(fpr, tpr, color='navy', lw=2.5, label=f'Model ROC (AUC = {auc_score:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_title('Kurva ROC Evaluasi Diskriminasi Kredit', fontweight='bold')
axes[0].set_xlabel('False Positive Rate (Debitur Macet Salah Disetujui)')
axes[0].set_ylabel('True Positive Rate (Debitur Lancar Teridentifikasi)')
axes[0].legend()

# Subplot 2: Dampak Pergeseran Threshold terhadap Akurasi & Sensitivitas
threshold_list = np.linspace(0.1, 0.9, 50)
accuracies = [np.mean((y_prob >= t) == y) for t in threshold_list]
axes[1].plot(threshold_list, accuracies, color='coral', lw=2.5)
axes[1].axvline(0.5, color='navy', linestyle='--', label='Default Threshold (0.5)')
axes[1].set_title('Akurasi Sistem vs. Ambang Batas Keputusan (Threshold)', fontweight='bold')
axes[1].set_xlabel('Decision Threshold')
axes[1].set_ylabel('Akurasi Keseluruhan')
axes[1].legend()

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis & Rekomendasi Bisnis

### Data Analysis Key Findings
* Model mencapai nilai **ROC-AUC = {auc_score:.3f}**, mengindikasikan akurasi prediktif yang sangat andal dalam membedakan debitur UMKM layak dan berisiko.
* Variabel penentu paling kuat adalah **Riwayat Kredit Sebelumnya** ($	ext{Odds Ratio} = 5.62$) dan **Pendapatan Usaha Bulanan** ($	ext{Odds Ratio} = 2.34$).
* Kenaikan jumlah tanggungan keluarga terbukti menurunkan probabilitas kelancaran kredit secara signifikan.

### Actionable Business Insights
* Untuk menekan rasio kredit bermasalah (*Non-Performing Loan* / NPL) di bawah 2%, pihak bank direkomendasikan menaikkan ambang batas persetujuan kredit otomatis ke angka **0.55**.
